In [1]:
from run_single_experiment import run_experiment_1, run_experiment_2, run_experiment_3, run_experiment_4

/Users/ukun/anaconda3/envs/CTGAN/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CWD = /Users/ukun/Desktop/captcha
captcha_data exists? True
secrets.yaml exists? True
sample type dirs: ['./captcha_data/Unusual_Detection', './captcha_data/Connect_icon', './captcha_data/Select_Animal', './captcha_data/Coordinates', './captcha_data/Geometry_Click']
{
  "providers": {
    "openai": {
      "api_key": "REDACTED_OPENAI_API_KEY"
    },
    "anthropic": {
      "api_key": "sk-ant-可选"
    },
    "gemini": {
      "api_key": "REDACTED_GEMINI_API_KEY"
    },
    "qwen": {
      "api_key": "sk-qwen-可选",
      "base_url": "https://dashscope.aliyuncs.com/compatible-mode/v1"
    }
  },
  "pricing": {
    "openai": {
      "gpt-4o-mini": {
        "in_per_1k": 0.15,
        "out_per_1k": 0.6
      }
    },
    "gemini": {
      "gemini-2.5-pro": {
        "in_per_1k": 0.00125,
        "out_per_1k": 0.01
      },
      "gemini-2.5-flash": {
        "in_per_1k": 0.0003,
        "out_per_1k": 0.0025
      },
      "gemini-2.5-flash-lite": {
        "in_per_1k": 0.0001,
        "out_p

In [ ]:
from pathlib import Path

RESULTS_BASE = Path("./results")
ERROR_BASE = Path("./error_analysis")

def _norm_model_name(model: str) -> str:
    return model.replace("/", "_")

def exp_results_dir(exp_name: str, provider: str, model: str) -> str:
    path = RESULTS_BASE / exp_name / provider.lower() / _norm_model_name(model)
    path.mkdir(parents=True, exist_ok=True)
    return str(path)

def exp_out_csv(exp_name: str, provider: str, model: str, filename: str = "results.csv") -> str:
    return str(Path(exp_results_dir(exp_name, provider, model)) / filename)

def exp_error_dir(exp_name: str, provider: str, model: str) -> str:
    path = ERROR_BASE / exp_name / provider.lower() / _norm_model_name(model)
    path.mkdir(parents=True, exist_ok=True)
    return str(path)


In [2]:
SUPPORTED_TYPES = {
    "Dice_Count",
    "Geometry_Click",
    "Image_Matching",
    "Patch_Select",
    "Place_Dot",

    "Bingo",
    "Click_Order",
    "Dart_Count",
    "Image_Recognition",
    "Misleading_Click",
    "Object_Match",
    "Path_Finder",
    "Pick_Area",
    "Select_Animal",
    "Unusual_Detection",
    "Coordinates",
    "Connect_Icon",
    "Rotation_Match",
}

ERROR_TYPES = {
    "Dice_Count",
    "Click_Order",
    "Patch_Select",
    "Place_Dot"
}

In [ ]:
# 实验一：Ground Truth Prompts
provider = "gemini"
model = "gemini-2.5-flash"

result1 = run_experiment_1(
    types=ERROR_TYPES,
    max_per_type=1,
    provider=provider,
    model=model,
    out_csv=exp_out_csv("exp1", provider, model),
    thinking=False,
    thinking_options={"budget": -1},
    collect_tokens=True,
    token_output_dir=exp_results_dir("exp1", provider, model),
    enable_error_analysis=True,
    error_analysis_dir=exp_error_dir("exp1", provider, model),
    collect_reasoning=True
)


In [ ]:
# 实验二：Optimized Prompts
provider = "gemini"
model = "gemini-2.5-pro"

result2 = run_experiment_2(
    types=ERROR_TYPES,
    prompts_file="./prompts_optimized.yaml",
    max_per_type=5,
    provider=provider,
    model=model,
    out_csv=exp_out_csv("exp2", provider, model),
    thinking=False,
    thinking_options={"budget": -1},
    collect_tokens=True,
    token_output_dir=exp_results_dir("exp2", provider, model),
    enable_error_analysis=True,
    error_analysis_dir=exp_error_dir("exp2", provider, model),
    collect_reasoning=True
)


In [ ]:
# 实验三：Until Correct
provider = "gemini"
model = "gemini-2.5-flash"

result3 = run_experiment_3(
    types=SUPPORTED_TYPES,
    prompts_file="./prompts_optimized.yaml",
    prompt_mode="auto",
    max_attempts_per_type=10,
    max_pool_per_type=10,
    out_csv=exp_out_csv("exp3", provider, model),
    provider=provider,
    model=model,
    thinking=True,
    thinking_options={"budget": -1},
    collect_tokens=True,
    token_output_dir=exp_results_dir("exp3", provider, model),
    collect_reasoning=True
)


In [4]:
# 实验四：Few-shot
provider = "gemini"
model = "gemini-2.5-flash"

result4 = run_experiment_4(
    prompts_file="./prompts_optimized.yaml",
    few_shot_file="./few_shot_examples.yaml",
    types=ERROR_TYPES,
    max_per_type=10,
    provider=provider,
    model=model,
    n_shot=2,
    thinking=False,
    enable_error_analysis=True,
    error_analysis_dir=exp_error_dir("exp4", provider, model),
    collect_tokens=True,
    token_output_dir=exp_results_dir("exp4", provider, model),
    collect_reasoning=True,
    out_csv=exp_out_csv("exp4", provider, model)
)



🟣 实验四：Optimized Prompts + Few-shot Learning
Provider: gemini
Model: gemini-2.5-flash
Prompts file: ./prompts_optimized.yaml
Few-shot file: ./few_shot_examples.yaml
N-shot: 2
Note: 每种类型使用前 2 个样本作为示例

[INFO] Few-shot learning enabled
[INFO] Excluding 8 few-shot examples from testing
[INFO] 将评测 32 道题（types={'Dice_Count', 'Place_Dot', 'Patch_Select', 'Click_Order'}）


Evaluating: 100% 32/32 [27:18<00:00, 51.20s/it]   


[SUMMARY by type]
- Click_Order        n=  8  pass@1=0.000  avg_e2e_ms=20335.7  avg_ttft_ms=20335.7  tokens_in=18848 tokens_out=1498 cost_usd=0.009399
- Dice_Count         n=  8  pass@1=0.125  avg_e2e_ms=20954.0  avg_ttft_ms=20954.0  tokens_in=12064 tokens_out=5494 cost_usd=0.017354
- Patch_Select       n=  8  pass@1=0.000  avg_e2e_ms=17523.9  avg_ttft_ms=17523.9  tokens_in=11673 tokens_out=1724 cost_usd=0.007812
- Place_Dot          n=  8  pass@1=0.000  avg_e2e_ms=145966.7  avg_ttft_ms=145966.7  tokens_in=12160 tokens_out=1313 cost_usd=0.006930
[INFO] Token summary written to ./results/exp4_g25_flash/exp4_fewshot_gemini_gemini-2.5-flash_token_summary.json

[OVERALL] tasks=32  pass@1=0.031  sum_e2e_ms=1638242.1  wall_ms=1638366.8
[OVERALL TOKENS] tokens_in=54745 tokens_out=10029 cost_usd=0.041496
[DONE] Pass@1 = 1/32 = 0.031 ; errors=0. 结果已保存到 ./results/exp4_g25_flash/results_exp4_g25_flash.csv
✅ 错误分析已保存到: ./error_analysis/exp4_g25_flash/exp4_fewshot
   - 错误案例: ./error_analysis/exp4_g